# AML LoT - TX table (patient pool, TX table, DOS and grace)

Split from `AML_LOT_VAL.ipynb`. Fully commented-out cells removed.

# SPARK SESSION

In [ ]:
import findspark
import pandas as pd
findspark.init ()
import pyspark
from pyspark.sql import SparkSession

spark= SparkSession.builder\
              .config('spark.executor.instances','20')\
              .config('spark.driver.memory','5g')\
              .config('spark.executor.memory','20g')\
              .config('spark.executor.cores','5')\
              .config('spark.executor.memoryOverhead','4g')\
              .appName("AML_KA")\
              .master("yarn")\
              .getOrCreate()

In [ ]:
spark.sparkContext.addPyFile("pldlib.egg")
from pldlib import stencil

# GLOBAL VARIABLES

In [ ]:
cws = 'Z_ABV_CWS_MABI_ONC_ANALYTICS' 

# HCP-CHILD
reltio_tbl = 'abv_mhcd_synd.mhcd_rltn_all_org_customr_wkly_tbl'
heme_account='{cws}.heme_account'.format(cws=cws) # UPLOADED BY COE

tx_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_tbl_final_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

sob_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_SOB_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
episode_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_EPISODE_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
regimen_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_BASIC_REGIMEN_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

patient_group = '{cws}.MABI_AML_REGIMEN_REVAMP_PATIENT_COHORT_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LoT IC INELIGIBLE
ic_inelig_final = '{cws}.MABI_AML_IC_INELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LOT IC ELIGIBLE
ic_elig_final = '{cws}.MABI_AML_IC_ELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws) # LOT table IC ELIG
pat_comb_lot = '{cws}.MABI_AML_PATS_COMB_LOT_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)  # UNION OF BOTH LOT FINAL TABLES
pat_comb_lot_rr = '{cws}.MABI_AML_PATS_LOT_RR_12M_UPD_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)    

pat_fst_dx_tx = '{cws}.MABI_AML_LOT_PAT_FST_DX_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

dx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_DX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx = '{cws}.MABI_AML_LOT_PAT_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig = '{cws}.MABI_AML_LOT_PAT_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

tx_lookback_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

lot_regimen_group = '{cws}.MABI_AML_PATS_LOT_GROUPING_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
patient_intensity_based_on_age_n_product = '{cws}.PATIENT_INTENSITY_BASED_ON_AGE_N_PRODUCT_v2'.format(cws=cws)

#HCP AFFILIATIONS
hcp_affiliations = '{cws}.MABI_AML_HCP_AFFILIATIONS'.format(cws=cws)

# LoT TABLE ELIGIBILITY CHECK
elig_flags_final='{cws}.MABI_AML_LOT_PAT_ELIG_FLAGS_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

#RR AML DX ONLY
aml_dx = ('C92.01', 'C92.02', 'C92.51', 'C92.52', 'C92.61', 'C92.62', 'C92.A1', 'C92.A2', 'C93.01', 'C93.02', 'C94.01', 'C94.21', 'C94.22')

#Product codes for Onureg, Inqovi, Rezlidhia and Vanflyta 

onureg_product_code =  spark.sql('''
                                        select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%AZACITIDINE%') 
          AND SOURCE_DRUG_FORM IN ('TAB','CAP')
                                        
                                        UNION ALL
                                 select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and UPPER(PROCEDURE_DESCription) LIKE ('%ONUREG%')
                                        
                                 ''') 
df1 = onureg_product_code.toPandas()
df2 = df1['PRODUCT_NDC_CODE'].values.tolist()
df3 = list(df2)
onureg_codes = str(df3)[1:-1]

inqovi_product_code =  spark.sql('''
                                        select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%DECITABINE/CEDAZURIDINE%') 
          AND SOURCE_DRUG_FORM IN ('TAB','CAP')        
                                 
                                       
                                        UNION ALL
                                        select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(PROCEDURE_DESCription) LIKE ('%INQOVI%') or UPPER(PROCEDURE_DESCription) LIKE ('%DECITABINE/CEDAZURIDINE%'))
                                 
                                       
                                ''') 
df4 = inqovi_product_code.toPandas()
df5 = df4['PRODUCT_NDC_CODE'].values.tolist()
df6 = list(df5)
inqovi_codes = str(df6)[1:-1]

rezlidhia_product_code =  spark.sql('''
                                        select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%OLUTASIDENIB%') or UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%REZLIDHIA%'))

                                        
                                        UNION ALL
                                    select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(PROCEDURE_DESCription) LIKE ('%REZLIDHIA%') or UPPER(PROCEDURE_DESCription) LIKE ('%OLUTASIDENIB%'))


                                ''') 
df7 = rezlidhia_product_code.toPandas()
df8 = df7['PRODUCT_NDC_CODE'].values.tolist()
df9 = list(df8)
rezlidhia_codes = str(df9)[1:-1]

vanflyta_product_code =  spark.sql('''  
                                   select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%QUIZARTINIB%') or UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%VANFLYTA%'))


                                         
                                        UNION ALL
                                   select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(PROCEDURE_DESCription) LIKE ('%VANFLYTA%') or UPPER(PROCEDURE_DESCription) LIKE ('%QUIZARTINIB%'))


                                ''') 
df10 = vanflyta_product_code.toPandas()
df11 = df10['PRODUCT_NDC_CODE'].values.tolist()
df12 = list(df11)
vanflyta_codes = str(df12)[1:-1]


#### New Patient Pool of AML 2+ Dx

New Code

In [ ]:
# New Patient Pool Created

patient_pool = spark.sql('''
SELECT distinct PATIENT_SK from
(
    select patient_sk, count(distinct mx_claim_id) as Count_of_claims
    FROM abv_val_ptd_onc_synd.dx_fact_curated_vw
    WHERE SOURCE_DIAGNOSIS_CODE_1 IN 
    (
        '205', '205.0', '205.00', '205.01', '205.02', 'C92', 'C92.9', 'C92.0', 'C92.00', 'C92.01', 
        'C92.02', 'C92.5', 'C92.50', 'C92.51', 'C92.52', 'C92.6', 'C92.60', 'C92.61', 'C92.62', 'C92.90', 
        'C92.A', 'C92.A0', 'C92.A1', 'C92.A2', 'C92.Z0', 'C93', 'C93.0', 'C93.00', 'C93.01', 'C93.02', 
        'C94.0', 'C94.00', 'C94.01', 'C94.2', 'C94.20', 'C94.21', 'C94.22'
    )
    and source_flag='SHA_PTD'
    group by 1
)
where count_of_claims>1
          ''')

patient_pool.createOrReplaceTempView('patient_pool')

# TX TABLE UPDATE

#### Creating the Tx Table

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {tx_tbl_name} '''.format(tx_tbl_name=tx_tbl_name))

Changes Made

In [ ]:
# Removed the Patient Indication Filter (Diagnosis Frequency, Recency, and Hierarchy criteria) and instead identify patients based on AML 2+ diagnosis.

spark.sql('''
CREATE TABLE {tx_tbl_name} AS
(
          select distinct patient_sk as patient_gid, PRODUCT_QTY_DISPENSED, PRODUCT_DAYS_SUPPLY, 
          mastered_product_name as final_product_name, claim_date, claim_id, MDM_NPI_NUMBER as npi_number, PROCEDURE_CODE as product_code, source_type as native_type
          FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
            where source_flag = 'SHA_PTD'
            and source_market = 'ONC'
            --and market_code = 'VENC_AML'   
            and patient_sk in (
                select patient_sk
                from patient_pool
            )
            and mastered_product_name IN (
                'AZACITIDINE','DAURISMO','DECITABINE','IDHIFA','L-DAC','TIBSOVO','VENCLEXTA',
                'XOSPATA','REZLIDHIA','MYLOTARG','RYDAPT','ARSENIC_TRIOXIDE','CLADRIBINE','CLOFARABINE','CYCLOPHOSPHAMIDE',
                'DAUNORUBICIN','ETOPOSIDE','FLUDARABINE','HI-DAC','IDARUBICIN','MITOXANTRONE','NEXAVAR','ONUREG','INQOVI',
                'S-DAC','VINCRISTINE','VANFLYTA','VYXEOS'
            )

)
'''.format(tx_tbl_name=tx_tbl_name))




In [ ]:
spark.sql(''' 
select count(distinct patient_gid) from
{tx_tbl_name}

'''.format(tx_tbl_name=tx_tbl_name)).show()

#### DOS AND GRACE ASSIGNMENT

In [ ]:
tx_tbl_final_name

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {tx_tbl_final_name}'''.format(tx_tbl_final_name=tx_tbl_final_name))

In [ ]:
spark.sql('''
CREATE TABLE {tx_tbl_final_name} AS
(
    SELECT *
    FROM
    (
        SELECT *,
        CASE
            WHEN FINAL_PRODUCT_NAME IN ('ARSENIC_TRIOXIDE', 'AZACITIDINE', 'DAURISMO', 'DECITABINE', 'IDHIFA', 'L-DAC', 'REZLIDHIA', 'RYDAPT', 'TIBSOVO', 'VENCLEXTA', 'XOSPATA') THEN 60
            WHEN FINAL_PRODUCT_NAME IN ('ONUREG', 'VINCRISTINE', 'INQOVI', 'VANFLYTA') THEN 28
            WHEN FINAL_PRODUCT_NAME IN ('NEXAVAR') THEN 18
            WHEN FINAL_PRODUCT_NAME IN ('CLADRIBINE', 'HI-DAC', 'MYLOTARG', 'S-DAC') THEN 7
            WHEN FINAL_PRODUCT_NAME IN ('CLOFARABINE', 'CYCLOPHOSPHAMIDE','DAUNORUBICIN', 'ETOPOSIDE', 'FLUDARABINE', 'IDARUBICIN', 'VYXEOS') THEN 5
            WHEN FINAL_PRODUCT_NAME IN ('MITOXANTRONE') THEN 3
            ELSE NULL
        END AS GRACE_VALUE
        FROM
        (
            SELECT *,
            CASE
                WHEN NATIVE_TYPE = 'RX' THEN 
                    CASE
                        WHEN (PRODUCT_DAYS_SUPPLY IS NULL OR PRODUCT_DAYS_SUPPLY <= 0) THEN 28
                        ELSE PRODUCT_DAYS_SUPPLY
                    END
                WHEN NATIVE_TYPE = 'PX' THEN
                    CASE
                        WHEN FINAL_PRODUCT_NAME = 'ARSENIC_TRIOXIDE' THEN 28 --28D CYCLE
                        WHEN FINAL_PRODUCT_NAME = 'AZACITIDINE' THEN 28 --28D CYCLE
                        WHEN FINAL_PRODUCT_NAME = 'CLADRIBINE' THEN 1
                        WHEN FINAL_PRODUCT_NAME = 'CLOFARABINE' THEN 1
                        WHEN FINAL_PRODUCT_NAME = 'CYCLOPHOSPHAMIDE' THEN 1
                        WHEN FINAL_PRODUCT_NAME IN ('L-DAC', 'S-DAC', 'HI-DAC') THEN 1
                        WHEN FINAL_PRODUCT_NAME = 'DAUNORUBICIN' THEN 1 --GRACE 5
                        WHEN FINAL_PRODUCT_NAME = 'DECITABINE' THEN 28 --28D CYCLE
                        WHEN FINAL_PRODUCT_NAME = 'ETOPOSIDE' THEN 1 --GRACE 5
                        WHEN FINAL_PRODUCT_NAME = 'FLUDARABINE' THEN 1
                        WHEN FINAL_PRODUCT_NAME = 'IDARUBICIN' THEN 1 --GRACE 5
                        WHEN FINAL_PRODUCT_NAME = 'MITOXANTRONE' THEN 1
                        WHEN FINAL_PRODUCT_NAME = 'MYLOTARG' THEN 1 --GRACE 7
                        WHEN FINAL_PRODUCT_NAME = 'VINCRISTINE' THEN 1 --GRACE 28
                        WHEN FINAL_PRODUCT_NAME = 'VYXEOS' THEN 1
                    END
            END AS DOS_FINAL
            FROM
            (
                SELECT DISTINCT PATIENT_GID, PRODUCT_QTY_DISPENSED, NPI_NUMBER,
                CASE
                    WHEN NATIVE_TYPE LIKE '%RX%' THEN 'RX'
                    ELSE 'PX'
                END AS NATIVE_TYPE,
                CLAIM_ID, CLAIM_DATE, PRODUCT_CODE, FINAL_PRODUCT_NAME AS FINAL_PRODUCT_NAME, 
                PRODUCT_DAYS_SUPPLY AS PRODUCT_DAYS_SUPPLY
                FROM {tx_tbl_name}
                WHERE FINAL_PRODUCT_NAME <> 'FILGRASTIM'
            )A
        )A
    )A
)    

'''.format(tx_tbl_name=tx_tbl_name, tx_tbl_final_name=tx_tbl_final_name))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID) AS NUM_PAT
FROM {tx_tbl_final_name}
'''.format(tx_tbl_final_name=tx_tbl_final_name)).toPandas().head(20)